In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import time
import os

# --- 1. JIT 极致加速：百亿步轨道与转移矩阵同步构建 ---
@njit(fastmath=True, nogil=True)
def compute_matrix_ultra(u_c, k, steps, n_bins):
    n_offset = 1e6
    x = 0.5
    # 充分热启动，确保系统进入稳态吸引子
    for i in range(500000):
        u = u_c - k * (np.log(i + n_offset))**-2
        x = 1 - u * x**2
    
    # 利用 480GB 内存优势，使用 float64 保证累加精度
    counts = np.zeros((n_bins, n_bins), dtype=np.float64) 
    last_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
    
    for i in range(steps):
        u = u_c - k * (np.log(i + n_offset))**-2
        x = 1 - u * x**2
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        if 0 <= current_bin < n_bins:
            counts[last_bin, current_bin] += 1
            last_bin = current_bin
    return counts

# --- 2. 核心计算任务：针对 AutoDL 节点优化 ---
def get_spectrum_task(k_val):
    U_C = 1.543689012  # 系统临界点
    STEPS = 10**10     # 百亿步暴力演化
    N_BINS = 12000     # 极致分辨率：12000x12000 矩阵
    
    t0 = time.time()
    # 执行动力学演化
    counts = compute_matrix_ultra(U_C, k_val, STEPS, N_BINS)
    
    # 算子构建与特征提取
    P = csr_matrix(counts)
    row_sums = np.array(P.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0
    P = P.multiply(1.0 / row_sums[:, np.newaxis])
    
    # 提取前 200 个主模式，捕捉高阶零点
    vals, _ = eigs(P, k=200, which='LM', tol=1e-7)
    phases = np.sort(np.angle(vals[np.abs(vals) > 0.6]))
    phases = phases[phases > 0.05]
    
    # 实时持久化 (CS 级备份)
    save_path = f"res_k_{k_val:.4f}_steps10t10.npy"
    np.save(save_path, phases)
    print(f"[PID {os.getpid()}] k={k_val:.4f} 完成 | 耗时: {time.time()-t0:.1f}s | 已存至: {save_path}")
    return k_val, phases[:10]

# --- 3. 分布式调度核心 ---
if __name__ == "__main__":
    # 扫描 64 个 k 值，充分利用 256 核资源 (每个任务占部分核心进行特征值求解)
    # 也可以设为 256 直接拉满扫描密度
    k_range = np.linspace(4.7, 12.73, 64) 
    
    print(f"开始暴力扫描 | 算力配置: 256核心/480GB | 任务数: {len(k_range)}")
    
    # AutoDL 上的进程池设置
    with mp.Pool(processes=64) as pool:
        results = pool.map(get_spectrum_task, k_range)

    print("\n>>> 全量数据扫描完成。请检查当前目录下 spectrum_k_*.npy 文件。")

开始暴力扫描 | 算力配置: 256核心/480GB | 任务数: 64
[PID 1796] k=11.2005 完成 | 耗时: 555.1s | 已存至: res_k_11.2005_steps10t10.npy
[PID 1757] k=6.2295 完成 | 耗时: 605.4s | 已存至: res_k_6.2295_steps10t10.npy
[PID 1794] k=10.9456 完成 | 耗时: 616.9s | 已存至: res_k_10.9456_steps10t10.npy
[PID 1797] k=11.3279 完成 | 耗时: 619.6s | 已存至: res_k_11.3279_steps10t10.npy
[PID 1798] k=11.4554 完成 | 耗时: 620.0s | 已存至: res_k_11.4554_steps10t10.npy
[PID 1750] k=5.3373 完成 | 耗时: 620.7s | 已存至: res_k_5.3373_steps10t10.npy
[PID 1767] k=7.5041 完成 | 耗时: 623.9s | 已存至: res_k_7.5041_steps10t10.npy
[PID 1769] k=7.7590 完成 | 耗时: 625.2s | 已存至: res_k_7.7590_steps10t10.npy
[PID 1795] k=11.0730 完成 | 耗时: 631.9s | 已存至: res_k_11.0730_steps10t10.npy
[PID 1768] k=7.6316 完成 | 耗时: 639.9s | 已存至: res_k_7.6316_steps10t10.npy
[PID 1748] k=5.0824 完成 | 耗时: 640.8s | 已存至: res_k_5.0824_steps10t10.npy
[PID 1747] k=4.9549 完成 | 耗时: 641.2s | 已存至: res_k_4.9549_steps10t10.npy
[PID 1787] k=10.0533 完成 | 耗时: 650.4s | 已存至: res_k_10.0533_steps10t10.npy
[PID 1761] k=6.7394 完成 | 耗时:

In [2]:
!pip install numba

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 9.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install scipy

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 10.9 MB/s eta 0:00:0000:0100:01
